# 15장 데이터셋과 DataLoader — 내 데이터를 배치로

**1단계 · 따라 하기** — 『법률가를 위한 인공지능 코딩 자습서』 실습 노트북

[![Colab에서 열기](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch15_1_follow.ipynb)

이 노트북에는 책의 코드가 순서대로 들어 있습니다. **위에서부터 셀을 하나씩 실행**하고(셀을 누른 뒤 `Shift+Enter`), 출력이 책과 같은지 대조하세요.

- 셀 아래에 미리 보이는 출력은 저자의 컴퓨터(CPU)에서 실행한 결과입니다. 걸린 시간은 환경마다 다릅니다.
- 책은 코드를 **붙여 넣지 말고 직접 치라**고 권합니다. 실행해 본 다음, 셀 아래에 빈 코드 셀을 만들어(`+ 코드`) 같은 코드를 눈으로 읽으며 쳐 보세요.
- 설명은 책에 있습니다. 이 노트북은 실행을 위한 것입니다.

| 단계 | 할 일 | 노트북 |
|---|---|---|
| 1. 따라 하기 | `make_shapes.py` 로 데이터를 만들고 `p01`~`p03` 을 치고 실행 | [ch15_1_follow.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch15_1_follow.ipynb) **(지금 이 노트북)** |
| 2. 주석 달기 | `ImageFolderDataset` 의 `__init__` 과 `__getitem__` 에 "언제 실행되는가(한 번 / 매 항목)"를 주석으로 | [ch15_2_comment.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch15_2_comment.ipynb) |
| 3. 다시 만들기 | 주석만 남긴 파일에서 `ImageFolderDataset` 복원. 그다음 14장 패키지의 `data/` 에 `shapes.py` 로 넣고 `--dataset shapes` 로 학습되게 등록 | [ch15_3_build.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch15_3_build.ipynb) |

In [ ]:
# 준비 셀 — 맨 먼저 한 번 실행하세요. 실습 코드와 데이터를 받아 옵니다.
import os, sys
if "google.colab" in sys.modules:                       # Colab 에서 열었을 때
    ROOT = "/content/AI-Coding-for-Lawyers"
    if not os.path.isdir(ROOT):
        !git clone -q --depth 1 https://github.com/BowonKwon/AI-Coding-for-Lawyers.git {ROOT}
else:                                                   # 내 컴퓨터에서 열었을 때 (저장소 폴더 안)
    ROOT = os.getcwd()
    while not os.path.isdir(os.path.join(ROOT, "notebooks")) and os.path.dirname(ROOT) != ROOT:
        ROOT = os.path.dirname(ROOT)
os.environ.setdefault("JUCOS_DATA", os.path.join(ROOT, "data"))   # 데이터를 두는 폴더
os.chdir(os.path.join(ROOT, "code", "15"))                      # 이 장의 코드 폴더로 이동
print("준비 끝. 작업 폴더: code/15")

준비 끝. 작업 폴더: code/15


## `make_shapes.py` — 15.3 실습용 이미지 폴더를 만든다: shapes/{circle,square,triangle}/*.png

이 파일은 준비 셀에서 이미 받아 두었습니다. 내용은 아래와 같습니다. (왼쪽 폴더 모양 아이콘에서 파일을 두 번 누르면 열어서 고칠 수도 있습니다.)

```python
# 15.3 실습용 이미지 폴더를 만든다: shapes/{circle,square,triangle}/*.png
# 실제 사진 대신 도형을 그려 쓰는 이유: 저작권·용량 문제가 없고, 무엇이 정답인지 분명하다.
import os
import random
from PIL import Image, ImageDraw

ROOT = os.path.join(os.environ.get("JUCOS_DATA", "./data"), "shapes")
CLASSES = ["circle", "square", "triangle"]
N_PER_CLASS = {"train": 200, "test": 50}
SIZE = 48


def draw(kind, rng):
    img = Image.new("RGB", (SIZE, SIZE), (255, 255, 255))
    d = ImageDraw.Draw(img)
    color = tuple(rng.randint(0, 180) for _ in range(3))
    r = rng.randint(10, 18)
    cx, cy = rng.randint(r + 2, SIZE - r - 2), rng.randint(r + 2, SIZE - r - 2)
    if kind == "circle":
        d.ellipse([cx - r, cy - r, cx + r, cy + r], fill=color)
    elif kind == "square":
        d.rectangle([cx - r, cy - r, cx + r, cy + r], fill=color)
    else:
        d.polygon([(cx, cy - r), (cx - r, cy + r), (cx + r, cy + r)], fill=color)
    return img


def main():
    rng = random.Random(0)
    total = 0
    for split, n in N_PER_CLASS.items():
        for kind in CLASSES:
            folder = os.path.join(ROOT, split, kind)
            os.makedirs(folder, exist_ok=True)
            for i in range(n):
                draw(kind, rng).save(os.path.join(folder, f"{kind}_{i:03d}.png"))
                total += 1
    print(f"{total}장 생성: {ROOT}/{{train,test}}/{{{','.join(CLASSES)}}}/")


if __name__ == "__main__":
    main()
```

In [ ]:
!python make_shapes.py

750장 생성: …\AI-Coding-for-Lawyers\data\shapes/{train,test}/{circle,square,triangle}/


## `p01_dataset_basics.py` — 15.1~15.2 Dataset 과 DataLoader 가 하는 일을 직접 본다

In [ ]:
# 15.1~15.2 Dataset 과 DataLoader 가 하는 일을 직접 본다
import os
import torch
from torch.utils.data import Dataset, DataLoader
from torchvision import datasets, transforms

DATA_ROOT = os.environ.get("JUCOS_DATA", "./data")

In [ ]:
# ---- torchvision 의 MNIST 도 결국 __len__ 과 __getitem__ 을 가진 Dataset 이다 (4장)
mnist = datasets.MNIST(DATA_ROOT, train=True, download=True, transform=transforms.ToTensor())
print("len:", len(mnist))
image, label = mnist[0]                          # __getitem__(0)
print("mnist[0] ->", type(image).__name__, image.shape, image.dtype, "| label", label, type(label).__name__)
print("픽셀 범위:", image.min().item(), "~", image.max().item())

len: 60000
mnist[0] -> Tensor torch.Size([1, 28, 28]) torch.float32 | label 5 int
픽셀 범위: 0.0 ~ 1.0


In [ ]:
# transform 을 빼면 무엇이 나오나
raw = datasets.MNIST(DATA_ROOT, train=True, download=True)          # transform 없음
img_raw, _ = raw[0]
print("transform 없이:", type(img_raw).__name__, img_raw.size, img_raw.mode)   # PIL 이미지 28x28, 흑백(L)

transform 없이: Image (28, 28) L


In [ ]:
# ---- 내가 만든 Dataset: 리스트를 감싸는 가장 작은 예
class SquaresDataset(Dataset):
    """i번째 항목이 (i를 담은 텐서, i의 제곱) 인 데이터셋. 데이터가 파일이 아니어도 된다."""

    def __init__(self, n):
        self.n = n

    def __len__(self):
        return self.n

    def __getitem__(self, i):
        if i < 0 or i >= self.n:
            raise IndexError(i)
        return torch.tensor([float(i)]), i * i


ds = SquaresDataset(10)
print("len:", len(ds), "| ds[3] ->", ds[3])

len: 10 | ds[3] -> (tensor([3.]), 9)


In [ ]:
# ---- DataLoader: 묶고(batch), 섞고(shuffle), 하나씩 꺼내 준다
loader = DataLoader(ds, batch_size=4, shuffle=False)
for step, (x, y) in enumerate(loader):
    print(f"배치 {step}: x.shape={tuple(x.shape)} y={y.tolist()}")   # 마지막 배치는 2개

loader = DataLoader(ds, batch_size=4, shuffle=True, drop_last=True)
torch.manual_seed(0)
for step, (x, y) in enumerate(loader):
    print(f"섞은 배치 {step}: y={y.tolist()}")                        # drop_last: 남는 2개는 버림

배치 0: x.shape=(4, 1) y=[0, 1, 4, 9]
배치 1: x.shape=(4, 1) y=[16, 25, 36, 49]
배치 2: x.shape=(2, 1) y=[64, 81]
섞은 배치 0: y=[36, 49, 1, 16]
섞은 배치 1: y=[4, 0, 81, 64]


In [ ]:
# ---- MNIST 를 배치로: 12장에서 매 걸음 받던 것이 바로 이것
loader = DataLoader(mnist, batch_size=64, shuffle=True)
images, labels = next(iter(loader))
print("MNIST 배치:", images.shape, labels.shape, "| labels[:8] =", labels[:8].tolist())
print("배치 수:", len(loader), "(=", len(mnist), "/ 64 올림)")

MNIST 배치: torch.Size([64, 1, 28, 28]) torch.Size([64]) | labels[:8] = [2, 7, 3, 1, 6, 7, 1, 6]
배치 수: 938 (= 60000 / 64 올림)


## `p02_custom_dataset.py` — 15.3~15.4 내 이미지 폴더를 Dataset 으로 만들고, transform 으로 전처리·증강한다

이 파일은 준비 셀에서 이미 받아 두었습니다. 내용은 아래와 같습니다. (왼쪽 폴더 모양 아이콘에서 파일을 두 번 누르면 열어서 고칠 수도 있습니다.)

```python
# 15.3~15.4 내 이미지 폴더를 Dataset 으로 만들고, transform 으로 전처리·증강한다
import os
import torch
from PIL import Image
from torch.utils.data import Dataset, DataLoader
from torchvision import transforms

ROOT = os.path.join(os.environ.get("JUCOS_DATA", "./data"), "shapes")


class ImageFolderDataset(Dataset):
    """폴더 이름이 정답인 이미지 폴더. root/{클래스}/{파일}.png"""

    def __init__(self, root, transform=None):
        self.classes = sorted(os.listdir(root))                        # ['circle', 'square', 'triangle']
        self.class_to_idx = {c: i for i, c in enumerate(self.classes)}  # 이름 → 번호
        self.items = []                                                # (경로, 번호) 목록
        for c in self.classes:
            folder = os.path.join(root, c)
            for fname in sorted(os.listdir(folder)):
                if fname.lower().endswith((".png", ".jpg", ".jpeg")):
                    self.items.append((os.path.join(folder, fname), self.class_to_idx[c]))
        self.transform = transform

    def __len__(self):
        return len(self.items)

    def __getitem__(self, i):
        path, label = self.items[i]
        image = Image.open(path).convert("RGB")                        # 파일 → PIL 이미지 (여기서 비로소 읽는다)
        if self.transform:
            image = self.transform(image)                              # PIL → 텐서 (+ 전처리)
        return image, label


if __name__ == "__main__":          # import 될 때는 아래 실험이 돌지 않게 (4장)
    # ---- transform 없이: 목록만 만든다. 이미지는 아직 안 읽는다
    ds = ImageFolderDataset(os.path.join(ROOT, "train"))
    print("클래스:", ds.classes, ds.class_to_idx)
    print("len:", len(ds), "| items[0] =", ds.items[0][0].replace(ROOT, "…"), ds.items[0][1])
    img, label = ds[0]
    print("ds[0] ->", type(img).__name__, img.size, img.mode, "| label", label)

    # ---- 기본 transform: 텐서로 바꾸고 표준화
    mean, std = (0.5, 0.5, 0.5), (0.5, 0.5, 0.5)
    basic = transforms.Compose([
        transforms.Resize((32, 32)),           # 크기 통일
        transforms.ToTensor(),                 # [3, 32, 32], 0~1
        transforms.Normalize(mean, std),       # (x - mean) / std → 대략 -1~1
    ])
    ds = ImageFolderDataset(os.path.join(ROOT, "train"), transform=basic)
    img, label = ds[0]
    print("transform 후:", img.shape, img.dtype, f"범위 {img.min():.2f}~{img.max():.2f}", "| label", label)

    # ---- 증강(augmentation): 학습 때만, 매번 다르게
    augment = transforms.Compose([
        transforms.Resize((32, 32)),
        transforms.RandomHorizontalFlip(),
        transforms.RandomRotation(15),
        transforms.ToTensor(),
        transforms.Normalize(mean, std),
    ])
    ds_aug = ImageFolderDataset(os.path.join(ROOT, "train"), transform=augment)
    torch.manual_seed(0)
    a, _ = ds_aug[0]
    b, _ = ds_aug[0]
    print("같은 이미지를 두 번 꺼내면 다른가:", not torch.equal(a, b))

    # ---- DataLoader 로 배치
    loader = DataLoader(ds, batch_size=32, shuffle=True)
    images, labels = next(iter(loader))
    print("배치:", images.shape, labels.shape, "| labels[:10] =", labels[:10].tolist())

    # ---- 클래스별 개수 세기 (3장의 집계 그대로)
    counts = {}
    for _, label in ds.items:
        counts[ds.classes[label]] = counts.get(ds.classes[label], 0) + 1
    print("클래스별 개수:", counts)
```

In [ ]:
!python p02_custom_dataset.py

클래스: ['circle', 'square', 'triangle'] {'circle': 0, 'square': 1, 'triangle': 2}
len: 600 | items[0] = …\train\circle\circle_000.png 0
ds[0] -> Image (48, 48) RGB | label 0
transform 후: torch.Size([3, 32, 32]) torch.float32 범위 -0.92~1.00 | label 0
같은 이미지를 두 번 꺼내면 다른가: True
배치: torch.Size([32, 3, 32, 32]) torch.Size([32]) | labels[:10] = [1, 1, 0, 2, 1, 2, 0, 0, 2, 0]
클래스별 개수: {'circle': 200, 'square': 200, 'triangle': 200}


## `p03_train_shapes.py` — 15.5 데이터만 바꿔 끼우기: 12장의 학습 루프에 도형 데이터셋을 넣는다. 모델은 MLP 그대로(입력 크기만 다름).

In [ ]:
# 15.5 데이터만 바꿔 끼우기: 12장의 학습 루프에 도형 데이터셋을 넣는다. 모델은 MLP 그대로(입력 크기만 다름).
import os
import time
import torch
import torch.nn as nn
from torch.utils.data import DataLoader
from torchvision import transforms

from p02_custom_dataset import ImageFolderDataset

ROOT = os.path.join(os.environ.get("JUCOS_DATA", "./data"), "shapes")
torch.manual_seed(0)

tf = transforms.Compose([transforms.Resize((32, 32)), transforms.ToTensor(),
                         transforms.Normalize((0.5,) * 3, (0.5,) * 3)])
train_set = ImageFolderDataset(os.path.join(ROOT, "train"), transform=tf)
test_set = ImageFolderDataset(os.path.join(ROOT, "test"), transform=tf)
train_loader = DataLoader(train_set, batch_size=32, shuffle=True)
test_loader = DataLoader(test_set, batch_size=64)


class MLP(nn.Module):
    def __init__(self, in_dim, hidden, out_dim):
        super().__init__()
        self.fc1 = nn.Linear(in_dim, hidden)
        self.fc2 = nn.Linear(hidden, out_dim)

    def forward(self, x):
        return self.fc2(torch.relu(self.fc1(x.flatten(1))))


model = MLP(3 * 32 * 32, 128, len(train_set.classes))     # 입력 3072, 출력 3 — 바뀐 것은 이 숫자뿐
loss_fn = nn.CrossEntropyLoss()
optimizer = torch.optim.SGD(model.parameters(), lr=0.05)


def accuracy(loader):
    model.eval()
    correct = 0
    with torch.no_grad():
        for x, y in loader:
            correct += (model(x).argmax(1) == y).sum().item()
    model.train()
    return correct / len(loader.dataset)


print(f"학습 {len(train_set)}장 / 시험 {len(test_set)}장 / 클래스 {train_set.classes}")
print(f"학습 전 정확도: {accuracy(test_loader):.3f}")
t0 = time.time()
for epoch in range(1, 11):
    for x, y in train_loader:
        optimizer.zero_grad()
        loss = loss_fn(model(x), y)
        loss.backward()
        optimizer.step()
    if epoch % 2 == 0:
        print(f"에포크 {epoch:2d}: 손실 {loss.item():.4f}  시험 정확도 {accuracy(test_loader):.3f}")
print(f"경과 {time.time() - t0:.0f}초")

학습 600장 / 시험 150장 / 클래스 ['circle', 'square', 'triangle']
학습 전 정확도: 0.327


에포크  2: 손실 0.7330  시험 정확도 0.553


에포크  4: 손실 0.6721  시험 정확도 0.653


에포크  6: 손실 0.6631  시험 정확도 0.647


에포크  8: 손실 0.6005  시험 정확도 0.593


에포크 10: 손실 0.6069  시험 정확도 0.447
경과 5초


## 다음 단계

- 2단계 「주석 달기」: [ch15_2_comment.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch15_2_comment.ipynb)
- 3단계 「다시 만들기」: [ch15_3_build.ipynb](https://colab.research.google.com/github/BowonKwon/AI-Coding-for-Lawyers/blob/main/notebooks/ch15_3_build.ipynb)
- 책의 「확인 문제」는 이 노트북 끝에 셀을 추가해 풀어 보세요.